In [1]:
%pwd

'/Users/niteshkr.jha/Desktop/Agentic AI Projects/WellnessAI/research'

In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'/Users/niteshkr.jha/Desktop/Agentic AI Projects/WellnessAI'

In [4]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/cg/5mr1myzx7k15hkdc69d4bh_h0000gn/T/ipykernel_76645/3754768683.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
/Users/niteshkr.jha/Desktop/Agentic AI Projects/WellnessAI/wellnessai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Extract data from pdf file

In [5]:
def load_pdf_file(data):
    loader = DirectoryLoader(data, glob= '*.pdf', loader_cls= PyPDFLoader)
    documents = loader.load()
    return documents

In [6]:
import os
print(os.getcwd())

/Users/niteshkr.jha/Desktop/Agentic AI Projects/WellnessAI


In [7]:
extracted_data = load_pdf_file("data")

In [8]:
print(len(extracted_data))

4505


In [9]:
print(extracted_data[0])

page_content='' metadata={'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'creator': 'Adobe Acrobat 6.0', 'creationdate': '2006-10-16T20:19:33+02:00', 'moddate': '2006-10-16T22:03:45+02:00', 'source': 'data/GEM_3rd_edition.pdf', 'total_pages': 4505, 'page': 0, 'page_label': 'i'}


## Chunking Document Objects

In [10]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [11]:
text_chunks = text_split(extracted_data=extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 40059


In [12]:
text_chunks[0]

Document(metadata={'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'creator': 'Adobe Acrobat 6.0', 'creationdate': '2006-10-16T20:19:33+02:00', 'moddate': '2006-10-16T22:03:45+02:00', 'source': 'data/GEM_3rd_edition.pdf', 'total_pages': 4505, 'page': 1, 'page_label': 'ii'}, page_content='The GALE\nENCYCLOPEDIA of\nMEDICINE\nTHIRD EDITION')

## Download the Embeddings from Hugging Face

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings

In [14]:
embeddings_model = download_hugging_face_embeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8224.13it/s]


In [15]:
query_result = embeddings_model.embed_query("Hello World")
print("length", len(query_result))

length 384


## Setting Up Pinecone DB

In [22]:
from dotenv import load_dotenv
import os

load_dotenv()  ## important for using .env file 

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
print(PINECONE_API_KEY)

pcsk_471MPp_8wZYqBr5Srpi1wrgZt8g8gr3avaS96GeNowkzH8xD1wQiSbaH8giZQAMsLYXZu2


In [25]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

pc = Pinecone(api_key= PINECONE_API_KEY)

index_name = "wellnessai"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        vector_type="dense",
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
        deletion_protection="disabled",
        tags={
            "environment": "development"
        }
    )

### Embed each chunk and upsert the embeddings into our pinecone index

In [26]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents = text_chunks,
    index_name = index_name,
    embedding = embeddings_model
)